# High-Resolution PCA Embedding Visualization

Previously DINOv2 could only produce high-resolution PCA videos with **FeatUp**.
With **DINOv2** we can extract high-resolution PCA embeddings directly.

This notebook lets you:
1. Load any image and extract DINOv2 patch features
2. Visualise the PCA components at multiple resolutions
3. Explore individual PCA components
4. Export a feature-map video (frame-by-frame PCA evolution)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from sklearn.decomposition import PCA
from torchvision import transforms as T
from torchvision.transforms import functional as TF

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Encoder

In [ ]:
from transformers import Dinov2Model

# Change to 'facebook/dinov2-large' for better features
encoder = Dinov2Model.from_pretrained('facebook/dinov2-base').to(device).eval()
PATCH_SIZE = 14
print(f'Encoder hidden size: {encoder.config.hidden_size}')

## 2. Extract Patch Features

In [ ]:
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def load_image(path, size=448):
    img = PILImage.open(path).convert('RGB')
    img = TF.resize(img, (size, size), T.InterpolationMode.BILINEAR)
    t = normalize(TF.to_tensor(img))
    return t.unsqueeze(0).to(device), np.array(img)

def extract_features(tensor):
    B, _, H, W = tensor.shape
    h, w = H // PATCH_SIZE, W // PATCH_SIZE
    with torch.no_grad():
        out = encoder(tensor)
        tokens = out.last_hidden_state[:, 1:]   # drop CLS
    C = tokens.shape[-1]
    return tokens.reshape(B, h, w, C).permute(0, 3, 1, 2)  # (B, C, h, w)

## 3. PCA Visualisation at Multiple Resolutions

In [ ]:
from src.utils import visualize_pca_features

# Replace with your image path
IMG_PATH = 'path/to/your/image.jpg'

sizes = [224, 336, 448]
fig, axes = plt.subplots(2, len(sizes), figsize=(6 * len(sizes), 10))

for col, size in enumerate(sizes):
    tensor, orig = load_image(IMG_PATH, size=size)
    features = extract_features(tensor)
    pca_rgb = visualize_pca_features(features, n_components=3)

    axes[0, col].imshow(orig)
    axes[0, col].set_title(f'Input ({size}×{size})', fontsize=12)
    axes[0, col].axis('off')

    axes[1, col].imshow(pca_rgb)
    h = size // PATCH_SIZE
    axes[1, col].set_title(f'PCA features ({h}×{h} patches)', fontsize=12)
    axes[1, col].axis('off')

plt.suptitle('DINOv2 PCA Features — Multi-Resolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. PCA Component Explorer

Each PCA component captures a different aspect of the scene.
Component 1 usually separates foreground/background.

In [ ]:
tensor, orig = load_image(IMG_PATH, size=448)
features = extract_features(tensor)     # (1, C, h, w)

C, h, w = features.shape[1:]
flat = features[0].permute(1, 2, 0).reshape(-1, C).cpu().float().numpy()

pca = PCA(n_components=6)
comps = pca.fit_transform(flat)         # (h*w, 6)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes[0, 0].imshow(orig); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')

# RGB composite
rgb = comps[:, :3].copy()
rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-8)
axes[0, 1].imshow(rgb.reshape(h, w, 3))
axes[0, 1].set_title('PCA RGB (comp 1-3)'); axes[0, 1].axis('off')

for i in range(6):
    ax = axes[(i + 2) // 4, (i + 2) % 4]
    comp = comps[:, i].reshape(h, w)
    ax.imshow(comp, cmap='RdBu_r')
    var = pca.explained_variance_ratio_[i] * 100
    ax.set_title(f'Component {i+1}  ({var:.1f}% var)')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Background Masking via PCA

The first PCA component reliably separates foreground from background — this gives a free coarse segmentation mask with zero training.

In [ ]:
import torch.nn.functional as F

# First component → foreground mask
comp1 = comps[:, 0].reshape(h, w)
threshold = np.median(comp1)           # simple median threshold
fg_mask = (comp1 > threshold).astype(np.float32)

# Upsample mask to image resolution
mask_t = torch.from_numpy(fg_mask).unsqueeze(0).unsqueeze(0)
mask_up = F.interpolate(mask_t, size=(448, 448), mode='nearest')[0, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig);                                   axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(mask_up, cmap='gray');                   axes[1].set_title('PCA foreground mask'); axes[1].axis('off')
masked = orig.copy(); masked[mask_up < 0.5] = 0
axes[2].imshow(masked);                                 axes[2].set_title('Masked image'); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 6. Export Feature-Map Video (optional)

Animate PCA features as the model sees different crops of a high-resolution image.

In [ ]:
import os

# Slide a 448×448 window across a larger image
BIG_IMAGE = 'path/to/large_image.jpg'   # e.g. a 1920×1080 photo
OUT_DIR   = 'frames/'
os.makedirs(OUT_DIR, exist_ok=True)

big = PILImage.open(BIG_IMAGE).convert('RGB')
W_big, H_big = big.size
WIN = 448
STEP = 64

frame_idx = 0
for top in range(0, H_big - WIN, STEP):
    for left in range(0, W_big - WIN, STEP):
        crop = TF.crop(big, top, left, WIN, WIN)
        t = normalize(TF.to_tensor(crop)).unsqueeze(0).to(device)
        feats = extract_features(t)
        pca_rgb = visualize_pca_features(feats)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(np.array(crop)); axes[0].set_title('Crop'); axes[0].axis('off')
        axes[1].imshow(pca_rgb);        axes[1].set_title('PCA'); axes[1].axis('off')
        plt.tight_layout()
        plt.savefig(f'{OUT_DIR}/frame_{frame_idx:04d}.png', dpi=80)
        plt.close()
        frame_idx += 1

print(f'Saved {frame_idx} frames to {OUT_DIR}')
print('Combine with: ffmpeg -r 10 -i frames/frame_%04d.png -c:v libx264 output.mp4')